## Prototype 1

In [3]:
import json

In [6]:
# fetch data
POST_DATA = '/Users/louisn/Project/Work/sierra-sml/research/data/x/post.json'
USER_DATA = '/Users/louisn/Project/Work/sierra-sml/research/data/x/user.json'

with open(POST_DATA, 'r') as f:
    post_data = [json.load(f)]    

with open(USER_DATA, 'r') as f:
    user_data = [json.load(f)]
    


### Post Data processing 

In [31]:
post_data

[{'_id': '69b0794b497afb0727cc2ab4',
  'id': '2f8ae394-13ea-4c7f-bc0a-a5c299f46bf2',
  'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
  'task_id': 'a2c3efd5-51c6-59a6-a798-641f12b73d13',
  'post_id': '2025960593480663257',
  'user_id': '1252667805947723776',
  'user_screen_name': 'txtdrjkt',
  'user_following_count': 29,
  'user_followers_count': 365499,
  'user_tweet_count': 25086,
  'user_verified': False,
  'hashtags': [],
  'urls': [],
  'user_mentions': [],
  'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
   'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg'],
  'full_text': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
  'in_reply_to_post_id': None,
  'in_reply_to_user_id': None,
  'in_reply_to_screen_name': None,
  'in_reply_to_following_count': 0,
  'in_reply_to_followers_count': 0,
  'in_reply_to_tweet_count': 0,
  'in_reply_to_verified': False,
  'in_quote_to_post_id': None,
  'in_quote_to_user_id': None,


Use cases:
1. Tabular data showing
2. Alerting System
3. Sentiment detection
4. Scoring System

In [ ]:

# Column to keep
COLUMNS_POST = [
    # Data Management 
    'post_id', # Unique post id (?)
    'objective_id', # To be joined on project_id. 

    # Main Displayed data 
    # Note: These columns will be used for TEXT filtering from frontend, so the db must be robust enough to support text search.
    # Final Name: USERNAME: Screen name of the user who created the post.
    'user_screen_name',
    # Final Name: CONTENT: The main content of the post.
    'full_text',


    # ANALYTICS 
    # Final Name: COMMENT: quote & reply -> Another user's share their opinion on the original post.
    'quote_count',
    'reply_count',

    # Final Name: SHARE: Retweet -> Another user shares the original post with their followers.
    'retweet_count',

    # Final Name: LIKE: Favorite -> Another user shows likes for the original post.
    'favourite_count',

    # Plotting on frontend 
    # Note: all platform warehouses should use the same format 
    # Final Name: POSTED_AT: Timestamp of post creation
    'post_created_at',


    # SCORING
    # User's Snapshoted Metadata
    # Final Name: Concated to be POST_METADATA JSONB -> Will be processed by other ETL Consumer for scoring
    'user_id',
    'user_following_count',
    'user_followers_count',
    'user_tweet_count',
    'user_verified',

    # Attachments 
    # Final Name: Concated to be POST_METADATA JSONB -> Will be processed by other ETL, Probably..
    'media_urls'
]

post = {k: v for k, v in post_data[0] .items() if k in COLUMNS_POST}

post

{'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'post_id': '2025960593480663257',
 'user_id': '1252667805947723776',
 'user_screen_name': 'txtdrjkt',
 'user_following_count': 29,
 'user_followers_count': 365499,
 'user_tweet_count': 25086,
 'user_verified': False,
 'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
  'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg'],
 'full_text': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
 'favourite_count': 963,
 'quote_count': 31,
 'reply_count': 32,
 'retweet_count': 51,
 'post_created_at': '2026-02-23T15:47:00'}

In [33]:
# SML analysis should be one layered analysis. 
# The connections between posts and users shouldn't be too important. 

# Process
processed = {
    # Data Management
    'post_id': post['post_id'],
    'objective_id': post['objective_id'],

    # Display
    'username': post['user_screen_name'],
    'content': post['full_text'],

    # Analytics
    'comment': post['quote_count'] + post['reply_count'],
    'share': post['retweet_count'],
    'like': post['favourite_count'],

    # Timestamp
    'posted_at': post['post_created_at'],

    # Metadata JSONB (for scoring pipeline)
    'post_metadata': {
        'user_metadata': {
            'user_id': post['user_id'],
            'user_following_count': post['user_following_count'],
            'user_followers_count': post['user_followers_count'],
            'user_tweet_count': post['user_tweet_count'],
            'user_verified': post['user_verified']
        },
        'media_urls': post['media_urls'],
    }
}

processed

{'post_id': '2025960593480663257',
 'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'username': 'txtdrjkt',
 'content': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
 'comment': 63,
 'share': 51,
 'like': 963,
 'posted_at': '2026-02-23T15:47:00',
 'post_metadata': {'user_metadata': {'user_id': '1252667805947723776',
   'user_following_count': 29,
   'user_followers_count': 365499,
   'user_tweet_count': 25086,
   'user_verified': False},
  'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
   'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg']}}

### User Data processing 

In [34]:
user_data

[{'_id': '69b0794b497afb0727cc2ac7',
  'id': '7d041ae3-5591-4a44-81e1-a76c9a70abd6',
  'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
  'task_id': 'a2c3efd5-51c6-59a6-a798-641f12b73d13',
  'user_id': '1252667805947723776',
  'name': 'TXT DARI JAKARTA',
  'screen_name': 'txtdrjkt',
  'is_blue_verified': True,
  'user_created_at': '2020-04-21T18:37:41',
  'user_description': 'Jakarta shitpost account, jangan serius-serius banget lahh | Biz & Media Partner: txtdrjkt@gmail.com',
  'user_description_urls': [],
  'user_description_location': None,
  'following_count': 29,
  'followers_count': 365499,
  'favourites_count': 681,
  'media_count': 8864,
  'possibly_sensitive': False,
  'total_tweet': 25086,
  'verified': False,
  'verified_type': None,
  'withheld_in_countries': [],
  'created_at': '2026-03-11T03:04:27'}]

Use Cases:
1. Anjing

In [ ]:
COLUMNS_USER = [
    # Data Management 
    'user_id', # Unique post id (?)
    'objective_id', # To be joined on project_id. 

    # 
]